# AdaBoost (Adaptive Boosting)

AdaBoost is a machine learning algorithm that belongs to the family of **ensemble methods**. It combines multiple weak learners (usually simple models like decision stumps) to create a strong classifier.

---

## 🧠 Core Idea

Instead of training one complex model, AdaBoost builds many small models **sequentially**, where each new model focuses more on the mistakes made by the previous ones.

---

## ⚙️ How It Works

1. **Initialize weights**  
   - Each training sample is assigned an equal weight.

2. **Train a weak learner**  
   - A simple model (e.g., decision stump) is trained on the data.

3. **Evaluate errors**  
   - The model’s performance is measured.
   - Misclassified points are identified.

4. **Update weights**  
   - Increase weights for misclassified points.
   - Decrease weights for correctly classified points.

5. **Repeat**  
   - Train the next weak learner using updated weights.

6. **Combine models**  
   - Final prediction is a weighted sum (or vote) of all weak learners.

---

## 📊 Key Characteristics

- Works well with **binary classification problems**
- Sensitive to **noisy data and outliers**
- Converts weak learners into a **strong learner**
- Emphasizes difficult samples progressively

---

## 🧩 Mathematical Intuition (Simplified)

Each weak learner is assigned a weight based on its accuracy:

- More accurate learners → higher influence
- Less accurate learners → lower influence


The final prediction in AdaBoost is computed as:

F(x) = sign( Σ (αₘ · hₘ(x)) ), for m = 1 to M

---

### 🔍 Where:

- **hₘ(x)** → the m-th weak learner (model)
- **αₘ (alphaₘ)** → weight assigned to the m-th learner based on its accuracy
- **M** → total number of weak learners

---

### 💡 Intuition

- Each weak learner votes on the prediction
- More accurate learners get **more influence (higher weight)**
- The final result is the **weighted majority vote**

---

## 📦 Advantages

- Simple and easy to implement
- Often achieves high accuracy
- No need for extensive parameter tuning
- Works with many types of weak learners

---

## ⚠️ Disadvantages

- Sensitive to noise and outliers
- Can overfit if data is too noisy
- Requires careful handling of weights

---

## 🧪 Common Use Cases

- Face detection (e.g., Viola-Jones algorithm)
- Spam filtering
- Text classification
- Fraud detection

---

## 🧭 Summary

AdaBoost is like a team of students taking a test:
- Each new student studies the questions others got wrong
- Over time, the group becomes very good at answering everything correctly

---


In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier

In [ ]:
#Create Simple Dataset
X = np.array([[1], [2], [3], [4], [5]])
y = np.array([1, 1, -1, -1, 1])  # labels: +1 / -1

In [4]:
#Initilize Weights
n_samples = X.shape[0]
print("Number of samples:", n_samples)
weights = np.ones(n_samples) / n_samples
print("Initial weights:", weights)

Number of samples: 5
Initial weights: [0.2 0.2 0.2 0.2 0.2]


In [ ]:
#Train Weak Learners Iteratively
n_estimators = 3  # number of weak learners
models = []
alphas = []

for t in range(n_estimators):
    print(f"\n--- Iteration {t+1} ---")

    # 1. Train weak learner
    stump = DecisionTreeClassifier(max_depth=1)
    # A decision stump is a simple decision tree with max_depth=1, which makes it a weak learner.
    stump.fit(X, y, sample_weight=weights)

    # 2. Predictions
    y_pred = stump.predict(X)

    # 3. Compute error
    error = np.sum(weights * (y != y_pred))
    print("Error:", error)

    # 4. Compute model weight (alpha)
    alpha = 0.5 * np.log((1 - error) / (error + 1e-10))
    print("Alpha:", alpha)

    # 5. Update sample weights
    weights = weights * np.exp(-alpha * y * y_pred)

    # Normalize weights
    weights = weights / np.sum(weights)
    print("Updated Weights:", weights)

    # Save model
    models.append(stump)
    alphas.append(alpha)


--- Iteration 1 ---
Error: 0.2
Alpha: 0.6931471803099453
Updated Weights: [0.125 0.125 0.125 0.125 0.5  ]

--- Iteration 2 ---
Error: 0.2500000000625
Alpha: 0.5493061439673882
Updated Weights: [0.25       0.25       0.08333333 0.08333333 0.33333333]

--- Iteration 3 ---
Error: 0.16666666675555553
Alpha: 0.8047189555970503
Updated Weights: [0.15 0.15 0.25 0.25 0.2 ]


In [6]:
#Final Prediction 
def predict(X, models, alphas):
    final_pred = np.zeros(X.shape[0])

    for alpha, model in zip(alphas, models):
        final_pred += alpha * model.predict(X)

    return np.sign(final_pred)

In [7]:
# Test the Model 
y_final = predict(X, models, alphas)
print("\nFinal Predictions:", y_final)


Final Predictions: [ 1.  1. -1. -1.  1.]


Scikit-Learn uses a multiclass version of AdaBoost called SAMME16 (which stands for Stagewise Additive Modeling using  
a Multiclass Exponential loss function).   
When there are just two classes, SAMME is equivalent to AdaBoost. If the predictors can estimate class probabilities  
(i.e., if they have a predict_proba() method), Scikit-Learn can use a variant of SAMME called SAMME.R (the R stands for “Real”),  
which relies on class probabilities rather than predictions and generally performs better.


In [ ]:
# from sklearn.ensemble import AdaBoostClassifier
# ada_clf = AdaBoostClassifier(
#  DecisionTreeClassifier(max_depth=1), n_estimators=200,
#  algorithm="SAMME.R", learning_rate=0.5)
# ada_clf.fit(X_train, y_train)
